# 🫀 퀘스트 46 · Q8-G1 — **환자 개인화 P 형태**의 진입 관문

| | **MedKOS / `notebooks/quest46_q8_g1_personal_pmorph.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층① 표현(Q7-AA 가 연 유일한 문) |
| 부모 런 | `quest46_q7aa_burden_target` · `quest46_q7p0_svdb_pdelin` |
| 범위 | **G0 · G1 · G4 · G5 · G7** 만. G2(라벨-이득 곡선)·G3(개인화 리듬 기저)·G6(부담 층화)은 **G1 통과 후** |

## 임상 주장이 Q7 과 다르다

- **Q7** = 「처음 보는 환자에서 SVEB 를 잡는다」 → 교차환자 효용은 **정밀한 0**(−0.0007)
- **Q8** = 「이 환자의 앞부분을 판독자가 라벨링한 뒤 **나머지를 자동 분류**한다」(홀터 반자동 판독)

Q7-AA 가 보인 건 부정이 아니라 **해리**다 — 환자 안 0.5608 [0.5170, 0.6097] ✅ / 환자 간 −0.0007.
「환자 안에는 있는데 환자를 못 건너간다」면 **환자 자신의 라벨로 학습**하면 쓸 수 있다.

## ★★★ 천장을 무엇과 비교하는가 (이 런의 가장 중요한 설계 결정)

사전등록된 진입 조건은 **AUROC > 0.6097** 이다. 그런데 **0.6097 은 그냥 AUROC 가 아니다** —
Q7-AA 의 AA1 은 **리듬 기저에 잔차화한 뒤 `f1` 층에서 매칭한 AUROC** 였다.

> **그러므로 학습 표현도 반드시 같은 통계량으로 재야 한다.**
> 안 그러면 P 창을 보면서 **RR 을 학습한** 모델이 천장을 거저 넘고, 우리는 「P 형태가 된다」는
> 잘못된 결론을 얻는다. 모델 입력에 RR 을 **넣지 않고**, 출력 점수를 **리듬 기저에 잔차화**한 뒤
> **매칭 AUROC** 를 낸다.

⚠️ 그리고 0.6097 은 **전 기록**·**고부담 23명**에서 나온 수인데, 이 런은 **뒤 절반(평가 구간)**·
**G1 코호트**에서 잰다. 모집단이 다르므로 **같은 구간에서 `p_score` 의 매칭 AUROC 를 함께 낸다**
(관문 아님 · 런 내 기준선). 사전등록 앵커는 0.6097 이고, 런 내 기준선은 해석용이다(R35 ①).

## 관문

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **G0** | 자산 항등(Q7-P0 `p_idx`·`p_score`) + **환자당 앞절반 S 비트 세기** | `(pid,sym)` 원소 일치 · 생리학적 타당성 · 코호트 미달이면 **중단** |
| **G1 ★★ 진입** | 학습 표현의 **잔차화·매칭 AUROC**, 3팔 | **> 0.6097**. 미달이면 딥러닝을 더 안 짓고 **종결** |
| **G4** | **파이프라인 영점** — 학습 라벨을 환자 안에서 치환하고 **재학습** | 0.5 를 **가정하지 말고 측정**(R26 · R38 ②) |
| **G5 ★★ 자 인공물** | `p_idx` 환자 내 셔플 팔 | **영점으로 떨어져야 한다** |
| **G7** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 3팔 (소거는 **판별력에서 한 번도 안 쟀다**)

```
raw          P 창 원신호
cancel       P 창 소거 잔차 (abs — 그 환자 **학습 구간**의 중앙 비트를 뺀다)
cancel+Pmask 같은 잔차인데 **P 봉우리 ±25ms 를 0 으로** → 음성 대조
```

소거에 대해 확정된 건 둘뿐이고 **판별력은 그중에 없다**: T2 ✅ 가시성 +0.2825 · U2 ❌ 분절 F1 −0.1984.
U2 가 죽인 건 **원 ECG 형태 사전을 가진 공개 분절기**이고, 환자 안에서 배우는 모델엔 그 사전이 없다.

⚠️ **T2 를 근거로 기대하지 마라**(R40 ①) — 「잘 보이게 하니 잘 맞힐 것」은 Q7-Z 가 정확히 그
패턴으로 죽은 추론이다. 소거는 **가설이 아니라 팔**이다.

⚠️ **소거 팔에만 걸리는 교란**: 템플릿은 다수 N 비트로 만들어지므로 S 비트의 잔차에는 P 뿐 아니라
**「소거가 안 맞는 정도」**가 섞인다. 그러면 모델이 **P 가 아니라 소거 실패**를 학습한다 →
`cancel+Pmask` 가 **반드시** 영점으로 떨어져야 한다. 이 프로젝트는 음성 대조가 표적 창을 이긴
전례가 **둘**(Q7-F F3 · `ailab-2026-0067`) 있으므로 **떨어질 거라 가정하지 않는다**.

★ `abs` 는 적합이 없는 **단순 중앙 비트 차감**이라 「적합 구간을 P 창과 겹치지 마라」(Q7-T 제약)가
적용되지 않는다. 대신 **P 창까지 차감된다** — 즉 이 팔이 재는 건 「그 환자의 중앙 비트 대비 편차」다.
그래서 음성 대조가 더더욱 필수다.

## 누출 차단

1. ★★ **시간 분할** — 앞 절반 학습 / 뒤 절반 평가 + **가드밴드 60초**. 환자 안 무작위 분할은
   누출이다(이웃 비트는 같은 잡음 실현·유도 접촉·호흡 위상을 공유한다).
2. ★★ **소거 템플릿도 학습 구간에서만** 만든다 — 전 기록으로 만들면 평가 구간 신호가 템플릿에 든다.
3. **모델 입력에 RR 을 넣지 않는다** — 위 천장 논의대로.
4. **용량 선택을 하지 않는다** — 진입 관문이므로 **고정된 작은 CNN 하나**만 쓴다(선택이 없으면
   선택 편의도 없다 · R22 · R36 ②). 용량 격자는 G1 을 넘은 뒤 G2 에서.
5. **새 데이터 0** — `svdb_data5.npz` + `svdb_pdelin.npz`.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    """★ **짝지은 차**(b − a) — 같은 환자에서 두 팔을 재므로 짝을 유지한다."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def _rank_avg(v):
    """동점에 **평균 순위**를 준다."""
    v = np.asarray(v, float); o = v.argsort()
    r = np.empty(len(v), float); r[o] = np.arange(len(v), dtype=float)
    for u in np.unique(v):
        m = v == u
        if m.sum() > 1:
            r[m] = r[m].mean()
    return r

def spearman(a, b):
    """★★ 동점을 **평균 순위**로 처리한다.

    ⚠️ 이전 런들이 쓰던 `argsort().argsort()` 판본은 동점을 **위치로** 깨뜨려
    **입력 순서에 따라 값이 달라졌다**(Q3-B 에서 발각: 같은 자료가 B4 +0.2496 · B5 +0.3880).
    무작위 순열 400회에서 서로 다른 ρ 가 126개 나왔다. 이 판본은 순서에 무관하다."""
    ra, rb = _rank_avg(a), _rank_avg(b)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)

# ── ★ 사전등록 문턱 (SMOKE 가 절대 안 건드린다)
CEIL = 0.6097            # ★★ 진입 앵커 — Q7-AA AA1 의 CI 상단(리듬 통제 후 매칭 AUROC)
FRAC_TRAIN = 0.50        # 앞 절반 학습
GUARD_S = 60.0           # 가드밴드 60초
MIN_S_SEG, MIN_N_SEG = 25, 25    # ★ 두 구간 **각각** 이만큼은 있어야 한다(GMIN_S 재현 · R11-b)
MIN_PAIR = 200           # 매칭 AUROC 최소 쌍
MIN_REC = 8              # 코호트가 이보다 얕으면 **중단**
HW_P = 32                # P 창 반폭(샘플) → 창 65샘플 ≈ 180ms
PMASK_MS = 25.0          # 음성 대조에서 0 으로 만들 P 봉우리 반폭

# ── 비용 손잡이(스모크에서만 축소)
NB_BOOT   = 400 if SMOKE else 3000
N_PERM_G4 = 1   if SMOKE else 5
N_SHUF_G5 = 1   if SMOKE else 3

# ★★ EPOCHS 는 **비용 손잡이가 아니라 설계 상수**다 — 스모크에서도 줄이지 않는다.
#    `fit_predict` 가 **풀배치**라 EPOCHS = 그래디언트 스텝 수다. 1판은 40 이었는데
#    합성 실측에서 40 스텝은 **덜 학습된** 상태였다(단일 환자 AUROC 40→0.7054 ·
#    200→0.7387 · 1000→0.7362 로 200 부근에서 평탄). 덜 학습된 모델로 관문을 읽으면
#    G1·G4·G5 가 전부 잡음이 된다(1판 스모크가 정확히 그렇게 죽었다).
EPOCHS = 200

ARMS = ("raw", "cancel", "cancel_pmask")
READ_ORDER = ("G0", "G1", "G4", "G5", "G7")
GATE_DEP = {"G1": ["G0"], "G4": ["G0"], "G5": ["G0", "G4"]}

SV5  = os.path.join(MITBIH, "svdb_data5.npz")
PDEL = os.path.join(MITBIH, "svdb_pdelin.npz")

REF = dict(
    aa1=0.5608, aa1_lo=0.5170, aa1_hi=0.6097, aa1_n=23, aa1_null=0.4991,
    aa5=-0.0007, aa5_lo=-0.0053, aa5_hi=0.0030,
    t2_visibility=0.2825, u2_delin=-0.1984,
    macro_auroc=0.8842)

RULE_CHECK = {
    "R11-b GMIN_S":     "★ 환자당 **구간별** S 비트를 세고 코호트를 그걸로 정한다 — 세기 전에 설계하지 않는다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 선택 없음":     "★★ 진입 관문이라 **용량 선택을 하지 않는다** — 고정된 작은 CNN 하나",
    "R24 / R27 기저":    "★★★ 모델 입력에 **RR 을 넣지 않고** 출력을 **리듬 기저에 잔차화**한다",
    "R26 / R38 ②":      "영점은 **측정**한다 — 0.5 를 가정하지 않는다",
    "R29 ② 분기 금지":   "G0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE 를 내고 점추정과 비교. **미결 ≠ 등가**",
    "R34 ③ 대조":       "★ `cancel_pmask` 는 **같은 창·같은 전처리**에 P 봉우리만 0 으로 — 구성 대조",
    "R35 ① 자 먼저":    "★★ 천장 0.6097 과 **같은 통계량**(잔차화·매칭)으로 잰다",
    "R40 ① 타당도":     "★★ T2(가시성)를 판별력의 근거로 삼지 않는다 — Q7-Z 가 그 패턴으로 죽었다",
    "누출 차단":        "★★ **시간 분할 + 가드밴드** · 소거 템플릿도 **학습 구간에서만**",
}

CONFIG = dict(
    exp="quest46_q8_g1_personal_pmorph", quest="ailab-2026-0046",
    step="personalized-p-morph-g1",
    parent_exp=["quest46_q7aa_burden_target", "quest46_q7p0_svdb_pdelin"],
    scope="G0 · G1 · G4 · G5 · G7 만. G2·G3·G6 은 G1 통과 후",
    purpose=("**Q8 진입 관문.** Q7-AA 의 해리(환자 안 0.5608 ✅ / 환자 간 −0.0007)가 연 문은 "
             "**환자 자신의 라벨로 학습해 그 환자 안에서 쓰는 모델**이다(홀터 반자동 판독). "
             "★★★ 천장 0.6097 은 **리듬 통제 후 매칭 AUROC** 이므로 학습 표현도 **같은 통계량**"
             "으로 잰다 — 모델 입력에 RR 을 넣지 않고, 출력을 리듬 기저에 잔차화한 뒤 f1 층에서 "
             "매칭한다. 안 그러면 RR 을 학습한 모델이 천장을 거저 넘는다. "
             "★★ 그리고 **QRST 소거를 팔로 넣는다** — 소거는 가시성(T2 +0.2825)과 분절 품질"
             "(U2 −0.1984)에서만 쟀고 **판별력에서는 한 번도 안 쟀다**. U2 가 죽인 건 원 ECG "
             "형태 사전을 가진 **공개 분절기**이고 환자 안에서 배우는 모델엔 그 사전이 없다. "
             "⚠️ 단 T2 를 근거로 기대하지 않는다(R40 ①) — 소거는 **가설이 아니라 팔**이다."),
    dataset="SVDB — svdb_data5.npz + svdb_pdelin.npz (새 데이터 0 · BUT PDB 불필요)",
    arms=list(ARMS), read_order=READ_ORDER, gate_dep=GATE_DEP,
    ceiling=CEIL, frac_train=FRAC_TRAIN, guard_s=GUARD_S,
    min_s_seg=MIN_S_SEG, min_n_seg=MIN_N_SEG, min_pair=MIN_PAIR, min_rec=MIN_REC,
    hw_p=HW_P, pmask_ms=PMASK_MS, epochs=EPOCHS,
    n_boot=NB_BOOT, n_perm_g4=N_PERM_G4, n_shuf_g5=N_SHUF_G5, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "G0": "자산 항등 + **환자당 구간별 S/N 세기**. 코호트 = 앞·뒤 구간 **각각** "
              f"S≥{MIN_S_SEG} & N≥{MIN_N_SEG} 인 환자. {MIN_REC}명 미만이면 **중단**하고 "
              "그 사실만 로그로 남긴다(설계를 세기에 의존시키지 않기 위해 코호트를 세기가 정한다)",
        "G1": f"★★ **진입 관문.** 학습 표현의 **잔차화·매칭 AUROC** 가 {CEIL} 을 넘는가. "
              "3팔(raw · cancel · cancel_pmask). 미달이면 **딥러닝을 더 짓지 않고 종결**한다. "
              "★ 같은 구간에서 `p_score` 의 같은 통계량을 **런 내 기준선**으로 함께 낸다(관문 아님)",
        "G4": f"**파이프라인 영점** — 학습 구간 라벨을 환자 안에서 치환하고 **재학습**한다"
              f"(reps={N_PERM_G4}). 통계량의 영점이 아니라 **학습 절차 전체**의 영점이다. "
              "0.5 를 가정하지 않는다",
        "G5": f"★★ **자 인공물** — `p_idx` 를 환자 안에서 셔플하고 같은 파이프라인을 밟는다"
              f"(reps={N_SHUF_G5}). **영점으로 떨어져야 한다.** 환자 안에서는 검출기 오차가 "
              "그 환자 고유라 모델이 **P 가 아니라 검출기 버릇**을 학습할 수 있다. "
              "★ 함께 **G5a(학습 없음)** — `p_idx` **그 자체**의 잔차화·매칭 AUROC 를 낸다. "
              "이게 검출기 버릇이 라벨과 상관되는지의 **직접 검정**이고(R40 ①), 셔플 팔의 "
              "해석을 가른다. ★ 앵커 **산포**도 함께 찍는다 — 창(±{}샘플)보다 산포가 훨씬 "
              "작으면 셔플이 창 내용을 거의 안 바꾸므로 그 대조는 **좁은 주장만** 검정한다"
              .format(HW_P),
        "G7": "결론 검산표 — 판정마다 (a) 근거 (b) 미검정 가정 (c) 틀리면"},
    caveat=("★★★ **G1 은 진입 관문이다** — 넘지 못하면 개인화라는 이름으로 되살리지 않는다. "
            "★★ **소거 팔의 교란**: 템플릿이 다수 N 비트로 만들어지므로 S 비트 잔차에는 "
            "「소거가 안 맞는 정도」가 섞인다 → 모델이 P 가 아니라 **소거 실패**를 학습할 수 있고, "
            "그래서 `cancel_pmask` 가 반드시 영점으로 떨어져야 한다. 이 프로젝트는 음성 대조가 "
            "표적 창을 이긴 전례가 **둘**(Q7-F F3 · ailab-2026-0067) 있으므로 **가정하지 않는다**. "
            "★ `abs` 소거는 적합이 없어 「적합 구간을 P 창과 겹치지 마라」가 적용되지 않는 대신 "
            "**P 창까지 차감된다** — 이 팔이 재는 건 「그 환자 중앙 비트 대비 편차」다. "
            "★ 0.6097 은 **전 기록·고부담 23명**의 수이고 이 런은 **뒤 절반·G1 코호트**에서 잰다 — "
            "모집단이 다르므로 런 내 `p_score` 기준선을 함께 읽는다. "
            "★ **반대 증거**: Q7-S′ 의 S1≈S2(가벼운 개인화는 이득 없었다) · Q7-Z(자를 고쳐도 안 올랐다)."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q8_g1_personal_pmorph", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q8-G1 — 환자 개인화 P 형태 진입 관문**")
run.log(f"  ★★★ 천장 {CEIL} 은 **리듬 통제 후 매칭 AUROC** — 학습 표현도 **같은 통계량**으로 잰다")
run.log(f"  ★★ 3팔 — {' · '.join(ARMS)}  (소거는 **판별력에서 한 번도 안 쟀다**)")
run.log(f"  ★ 시간 분할 앞 {FRAC_TRAIN:.0%} 학습 / 뒤 평가 · 가드밴드 {GUARD_S:.0f}초")
run.log("  ★ 용량 선택 없음 — 진입 관문이라 **고정된 작은 CNN 하나**")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(EPOCHS={EPOCHS} · G4 reps={N_PERM_G4} · "
            f"G5 reps={N_SHUF_G5} · NB_BOOT={NB_BOOT}). 관문 문턱은 그대로다")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【G-0】 자산 항등 · 시간 분할 · ★ 환자당 구간별 S 세기 · 코호트
import pandas as pd
run.log("\n" + "=" * 100)
run.log("【G-0】 자산 항등 · 시간 분할 · **세기** · 코호트")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "Q7-P0")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True); PD = np.load(PDEL, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); SYM = np.asarray(D5["sym"]).astype(str)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)

# ── ★ 구성적 정합 증명 (Q7-P0 규약) — 비싼 계산 **전에** 끝낸다(R35 ⑦)
pid2 = np.asarray(PD["pid"]).astype(int); sym2 = np.asarray(PD["sym"]).astype(str)
if len(pid2) != len(PID):
    raise AssetError(f"길이 불일치 — d5 {len(PID)} vs pdelin {len(pid2)}")
bad = np.where((pid2 != PID) | (sym2 != SYM))[0]
if len(bad):
    raise AssetError(f"정합 깨짐 — 첫 불일치 idx {int(bad[0])} "
                     f"(d5 {PID[bad[0]]}/{SYM[bad[0]]} vs pdelin {pid2[bad[0]]}/{sym2[bad[0]]})")
run.log(f"  자산 정합 ✅ (pid, sym) 원소 단위 일치 — {len(PID):,} 비트")

P_IDX = np.asarray(PD["p_idx"]).astype(int)
P_SC = np.asarray(PD["p_score"], float)
R_SMP = np.asarray(PD["r_samp"], float)

K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx0 = P_IDX[K].copy(); psc_asset = P_SC[K].copy(); rsmp = R_SMP[K].copy()
XB = np.ascontiguousarray(np.asarray(D5["beat"])[K][:, 0, :]).astype(float)   # 유도 0
RS = np.array(sorted(set(RID.tolist())))
run.log(f"  채점 대상 비트 {len(K):,} · 레코드 {len(RS)}")

# ── ★ 생리학적 자기검증 — P 위치 중앙값이 PR 120~200ms 근처여야 한다(R35 ⑦)
FIRE = pidx0 >= 0
pr_ms_all = (RPRE - pidx0[FIRE]) / FS * 1000.0
pr_med = float(np.median(pr_ms_all))
run.log(f"  P 검출 {FIRE.mean():.3f} · PR 중앙 **{pr_med:.1f}ms** (타당 범위 100~230)")
if not (100.0 <= pr_med <= 230.0):
    raise AssetError(f"P 좌표가 생리학적으로 말이 안 된다(PR 중앙 {pr_med:.1f}ms)")

# ── 리듬 특징 (Q7-AA 와 동일) — ★ 모델 입력이 아니라 **잔차화·매칭용**이다
_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))

def basis_ext(idx):
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    """★★ 리듬 기저에 잔차화 — 천장 0.6097 과 **같은 통제**(Q7-AA)."""
    X = basis_ext(idx); y = v[idx] if len(v) == len(RID) else np.asarray(v, float)
    okm = np.isfinite(y)
    if okm.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[okm], y[okm], rcond=None)[0]
    return y - X @ b

def matched_auc(vsub, idx, perm=None, rng=None):
    """★★ `f1` 층 안에서만 쌍을 만든다 — 천장 0.6097 과 **같은 통계량**(Q7-AA)."""
    tt = TT[idx]; key = np.round(f1[idx]).astype(int)
    if perm == "stratum":
        tt = tt.copy()
        for kk in np.unique(key):
            m = np.where(key == kk)[0]
            tt[m] = tt[m][rng.permutation(len(m))]
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return ((win + 0.5 * tie) / tot, int(tot)) if tot >= MIN_PAIR else (float("nan"), int(tot))

# ── ★★ 시간 분할 — 앞 절반 학습 / 뒤 평가 + 가드밴드 (환자 안 무작위 분할은 누출이다)
t_sec = rsmp / FS
TRN, EVL = {}, {}
rows = []
for r in RS:
    ii = np.where(RID == r)[0]
    t = t_sec[ii]; t0, t1 = t.min(), t.max()
    cut = t0 + FRAC_TRAIN * (t1 - t0)
    tr = ii[t < cut - GUARD_S / 2.0]
    ev = ii[t > cut + GUARD_S / 2.0]
    TRN[int(r)], EVL[int(r)] = tr, ev
    rows.append(dict(rec=int(r), dur_min=float((t1 - t0) / 60.0),
                     tr_s=int(TT[tr].sum()), tr_n=int((~TT[tr]).sum()),
                     ev_s=int(TT[ev].sum()), ev_n=int((~TT[ev]).sum()),
                     burden=float(TT[ii].mean())))

# ── ★★★ 세기가 코호트를 정한다 (설계를 세기에 의존시키지 않기 위해 · R11-b)
OKR = [d for d in rows if d["tr_s"] >= MIN_S_SEG and d["tr_n"] >= MIN_N_SEG
       and d["ev_s"] >= MIN_S_SEG and d["ev_n"] >= MIN_N_SEG]
COH = [d["rec"] for d in OKR]
run.log(f"\n  ★ 앞{FRAC_TRAIN:.0%}/뒤 시간 분할 · 가드밴드 {GUARD_S:.0f}초 · 기록 길이 중앙 "
        f"{np.median([d['dur_min'] for d in rows]):.1f}분")
run.log(f"  ★★ **세기** — 구간별 S≥{MIN_S_SEG} & N≥{MIN_N_SEG} 를 만족하는 환자 "
        f"**{len(COH)}명** / {len(rows)}명")
sv = np.array([d["tr_s"] for d in rows], float)
run.log(f"     학습구간 S 분포 — 중앙 {np.median(sv):.0f} · 사분위 "
        f"[{np.percentile(sv,25):.0f}, {np.percentile(sv,75):.0f}] · 최대 {sv.max():.0f}")
run.log(f"  {'rec':>5}{'분':>7}{'학습S':>7}{'학습N':>7}{'평가S':>7}{'평가N':>7}{'부담':>8}")
for d in sorted(OKR, key=lambda x: -x["burden"])[:10]:
    run.log(f"  {d['rec']:>5}{d['dur_min']:>7.1f}{d['tr_s']:>7}{d['tr_n']:>7}"
            f"{d['ev_s']:>7}{d['ev_n']:>7}{d['burden']:>8.4f}")
if len(COH) < MIN_REC:
    raise AssetError(f"코호트가 {len(COH)}명뿐이다(<{MIN_REC}) — G1 이 성립하지 않는다. "
                     "세기 결과만 로그로 남기고 **여기서 종결**한다(R11-b)")
g_("G0", "✅ 지지",
   f"자산 정합·생리학적 타당성 통과 · 시간 분할 코호트 **{len(COH)}명** 확보")
CONFIG["G0"] = dict(n_beat=int(len(K)), n_rec=int(len(RS)), pr_med=pr_med,
                    p_fire=float(FIRE.mean()), cohort=COH, rows=rows,
                    n_cohort=len(COH))
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【G-A】 3팔 창 구성 (raw · cancel · cancel_pmask) · 모델 · 런 내 기준선
import torch
import torch.nn as nn
run.log("\n" + "=" * 100)
run.log("【G-A】 3팔 창 구성 · 고정 CNN · **런 내 `p_score` 기준선**")
run.log("=" * 100)
DEV_T = "cuda" if torch.cuda.is_available() else "cpu"
run.log(f"  torch {torch.__version__} · device {DEV_T}")

PMASK_HW = int(round(PMASK_MS * FS / 1000.0))
W = 2 * HW_P + 1
run.log(f"  P 창 {W}샘플({W/FS*1000:.0f}ms) · 음성 대조 마스크 ±{PMASK_HW}샘플"
        f"({PMASK_MS:.0f}ms)")

def anchors(rec):
    """P 봉우리 앵커. 미검출 비트는 **그 환자 N 비트의 중앙 앵커**로 대체한다(사전등록)."""
    ii = np.concatenate([TRN[rec], EVL[rec]])
    a = pidx0[ii].astype(float)
    good = a[a >= 0]
    fill = float(np.median(good)) if len(good) else float(RPRE - 0.16 * FS)
    out = {int(k): (int(a[j]) if a[j] >= 0 else int(round(fill))) for j, k in enumerate(ii)}
    return out, fill

def windows(rec, idx, arm, anc, template=None):
    """★ 세 팔이 **정확히 같은 창 위치**를 본다 — 신호만 다르다(짝지은 비교)."""
    X = XB[idx]
    if arm != "raw":
        if template is None:
            raise AssetError("소거 팔인데 템플릿이 없다")
        X = X - template[None, :]        # ★ abs 소거 — 학습 구간 중앙 비트를 뺀다
    out = np.zeros((len(idx), W), float)
    for j, gi in enumerate(idx):
        c = anc[int(gi)]
        lo, hi = c - HW_P, c + HW_P + 1
        a, b = max(lo, 0), min(hi, X.shape[1])
        out[j, a - lo:b - lo] = X[j, a:b]
    if arm == "cancel_pmask":
        out[:, HW_P - PMASK_HW:HW_P + PMASK_HW + 1] = 0.0   # ★ P 봉우리만 0
    return out

class TinyCNN(nn.Module):
    """★ **고정 구조**(용량 선택 없음 · 진입 관문이므로). 약 740 파라미터."""
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(1, 8, 7, padding=3)
        self.c2 = nn.Conv1d(8, 16, 5, padding=2)
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        h = torch.relu(self.c1(x))
        h = torch.max_pool1d(h, 2)
        h = torch.relu(self.c2(h))
        h = torch.mean(h, dim=2)
        return self.fc(h).squeeze(1)

def fit_predict(Xtr, ytr, Xev, seed):
    """환자 한 명 · 팔 하나. 표준화는 **학습 구간 통계**로만(R22)."""
    torch.manual_seed(seed); np.random.seed(seed % (2**31))
    mu, sd = Xtr.mean(), Xtr.std() + 1e-9
    xt = torch.tensor((Xtr - mu) / sd, dtype=torch.float32, device=DEV_T).unsqueeze(1)
    xe = torch.tensor((Xev - mu) / sd, dtype=torch.float32, device=DEV_T).unsqueeze(1)
    yt = torch.tensor(ytr, dtype=torch.float32, device=DEV_T)
    npos, nneg = float(ytr.sum()), float((1 - ytr).sum())
    pw = torch.tensor(max(nneg, 1.0) / max(npos, 1.0), dtype=torch.float32, device=DEV_T)
    m = TinyCNN().to(DEV_T)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    m.train()
    for _ in range(EPOCHS):
        opt.zero_grad(); loss = lossf(m(xt), yt); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return m(xe).cpu().numpy().astype(float)

def stat_of(rec, ev, score_ev):
    """★★★ 천장과 **같은 통계량** — 리듬 잔차화 후 f1 층 매칭 AUROC."""
    v = np.full(len(RID), np.nan)
    v[ev] = score_ev
    return matched_auc(resid(v, ev), ev)

# ── ★ 런 내 기준선 — 같은 구간에서 `p_score` 의 같은 통계량 (관문 아님)
BASE = {}
for rec in COH:
    ev = EVL[rec]
    a, npair = stat_of(rec, ev, psc_asset[ev])
    BASE[rec] = dict(auc=a, pair=npair)
bm, blo, bhi, bn = boot_mean([BASE[r]["auc"] for r in COH], SEED0 + 21, NB_BOOT)
run.log(f"\n  ★ 런 내 기준선 — 평가 구간 `p_score` 매칭 AUROC **{bm:.4f}** "
        f"[{blo:.4f}, {bhi:.4f}] · n={bn}")
run.log(f"     (사전등록 앵커 {CEIL} 은 **전 기록·고부담 23명**의 수 — 모집단이 다르다. "
        "관문은 앵커로, 해석은 둘을 나란히)")
CONFIG["baseline_inrun"] = dict(mean=bm, lo=blo, hi=bhi, n=int(bn))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【G-B】 ★★ G1 — 진입 관문. 3팔 학습 표현
run.log("\n" + "=" * 100)
run.log("【G-B】 G1 — **진입 관문**: 학습 표현이 천장을 넘는가 (3팔)")
run.log("=" * 100)
run.log("  ▸ 모델 입력에 **RR 이 없다** · 출력을 **리듬 기저에 잔차화**한 뒤 f1 층 매칭")
run.log("  ▸ 세 팔은 **정확히 같은 창 위치**를 본다 — 신호만 다르다")

def run_patient(rec, arm, perm_train=None, shuf_anchor=None, seed=0):
    """환자 한 명 · 팔 하나를 끝까지 돌린다.
    perm_train  : G4 — 학습 라벨을 환자 안에서 치환한 뒤 **재학습**
    shuf_anchor : G5 — `p_idx` 를 환자 안에서 셔플"""
    tr, ev = TRN[rec], EVL[rec]
    anc, _ = anchors(rec)
    if shuf_anchor is not None:
        keys = list(anc.keys())
        vals = [anc[k] for k in keys]
        vals = [vals[i] for i in shuf_anchor.permutation(len(vals))]
        anc = {k: v for k, v in zip(keys, vals)}
    tmpl = None
    if arm != "raw":
        # ★★ 템플릿도 **학습 구간에서만** — 전 기록으로 만들면 평가 구간이 템플릿에 든다
        tmpl = np.median(XB[tr], axis=0)
    Xtr, Xev = windows(rec, tr, arm, anc, tmpl), windows(rec, ev, arm, anc, tmpl)
    ytr = TT[tr].astype(float)
    if perm_train is not None:
        ytr = ytr[perm_train.permutation(len(ytr))]
    sc = fit_predict(Xtr, ytr, Xev, seed)
    return stat_of(rec, ev, sc)

T0 = time.time()
G1 = {}
for arm in ARMS:
    per = []
    for rec in COH:
        a, _ = run_patient(rec, arm, seed=SEED0 + 1000 + rec)
        per.append(a)
    m_, lo_, hi_, n_ = boot_mean(per, SEED0 + 31, NB_BOOT)
    G1[arm] = dict(per={int(r): float(v) for r, v in zip(COH, per)},
                   mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
    run.log(f"  {arm:<14} 매칭 AUROC **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · n={n_} · "
            f"MDE {mde(lo_, hi_):.4f}")
run.log(f"  ({time.time()-T0:.0f}초)")

# ── 판정: 사전등록 앵커 CEIL
run.log(f"\n  ★★ 판정 — 사전등록 앵커 **{CEIL}** (Q7-AA AA1 의 CI 상단)")
for arm in ARMS:
    d = G1[arm]
    run.log(f"    {arm:<14}{decide(d['lo'], d['hi'], CEIL, '>')}  "
            f"({d['mean']:.4f} vs {CEIL})")
BEST = max(ARMS, key=lambda a: G1[a]["mean"])
g1_v = decide(G1[BEST]["lo"], G1[BEST]["hi"], CEIL, ">")
g_("G1", g1_v,
   f"최량 팔 **{BEST}** {G1[BEST]['mean']:.4f} [{G1[BEST]['lo']:.4f}, {G1[BEST]['hi']:.4f}] "
   f"vs 앵커 {CEIL}" + ("" if g1_v.startswith("✅") else " — **미달이면 종결한다**"))
run.log(f"  ⚠️ 최량 팔은 **사후 선택**이다 — 팔별 판정을 위에 전부 찍었고, 종결 판단은 "
        "**어느 팔도 못 넘을 때**만 한다(선택 편의 방지 · R36 ②)")

# ── 소거 팔의 짝지은 대비 (같은 환자)
run.log("\n  소거 대비 (**짝지은 차** · 같은 환자)")
PAIRS = {}
for a, b in (("raw", "cancel"), ("cancel_pmask", "cancel")):
    va = [G1[a]["per"][r] for r in COH]; vb = [G1[b]["per"][r] for r in COH]
    pm, plo, phi, pn = boot_pair(va, vb, SEED0 + 41, NB_BOOT)
    PAIRS[f"{b}−{a}"] = dict(mean=pm, lo=plo, hi=phi, n=int(pn))
    run.log(f"    {b} − {a:<14} {pm:+.4f} [{plo:+.4f}, {phi:+.4f}] · n={pn}")
run.log("    ▸ `cancel − cancel_pmask` 가 **소거 잔차 중 P 봉우리의 몫**이다")
CONFIG["G1"] = {k: {kk: vv for kk, vv in v.items() if kk != "per"} for k, v in G1.items()}
CONFIG["G1_per"] = {k: v["per"] for k, v in G1.items()}
CONFIG["G1_pairs"] = PAIRS; CONFIG["G1_best"] = BEST
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【G-C】 G4 파이프라인 영점 · ★★ G5 `p_idx` 셔플
run.log("\n" + "=" * 100)
run.log("【G-C】 G4 — **파이프라인 영점**(학습 라벨 치환 + 재학습) · G5 — `p_idx` 셔플")
run.log("=" * 100)
run.log("  ▸ 통계량의 영점이 아니라 **학습 절차 전체**의 영점이다 — 0.5 를 가정하지 않는다")

T1 = time.time()
G4, G5 = {}, {}
for arm in ARMS:
    nul = []
    for rep in range(N_PERM_G4):
        for rec in COH:
            pr = np.random.RandomState(SEED0 + 5000 + 97 * rep + rec)
            a, _ = run_patient(rec, arm, perm_train=pr, seed=SEED0 + 6000 + 97 * rep + rec)
            nul.append(a)
    m_, lo_, hi_, n_ = boot_mean(nul, SEED0 + 51, NB_BOOT)
    G4[arm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_))
    run.log(f"  G4 {arm:<14} 영점 **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · n={n_}")
run.log(f"  ({time.time()-T1:.0f}초 · reps={N_PERM_G4})")
g4_ok = all(np.isfinite(G4[a]["mean"]) for a in ARMS)
g_("G4", "✅ 지지" if g4_ok else "⚠️ 미결",
   "영점을 **측정**했다 — 아래 판정은 0.5 가 아니라 이 값 기준으로 읽는다" if g4_ok else
   "영점이 안 섰다 — 아래를 읽지 않는다")

# ── ★ G5a — **학습 없음.** `p_idx` 그 자체가 라벨과 상관되는가(검출기 버릇의 직접 검정)
run.log("\n  ★ G5a — `p_idx` **그 자체**의 매칭 AUROC (학습 없음 · 검출기 버릇 직접 검정)")
g5a, disp = [], []
for rec in COH:
    ev = EVL[rec]
    anc, _ = anchors(rec)
    v = np.full(len(RID), np.nan)
    v[ev] = np.array([anc[int(g)] for g in ev], float)
    a, _ = matched_auc(resid(v, ev), ev)
    g5a.append(a)
    allv = np.array([anc[int(g)] for g in np.concatenate([TRN[rec], EVL[rec]])], float)
    disp.append(float(allv.std()))
am, alo, ahi, an = boot_mean(g5a, SEED0 + 71, NB_BOOT)
DISP = float(np.median(disp))
run.log(f"    `p_idx` 단독 매칭 AUROC **{am:.4f}** [{alo:.4f}, {ahi:.4f}] · n={an}")
run.log(f"    앵커 환자 내 산포(중앙 SD) **{DISP:.1f}샘플** = 창 반폭 {HW_P}샘플의 "
        f"**{DISP/HW_P:.0%}**")
if DISP < 0.25 * HW_P:
    run.log("    ⚠️ **산포가 창보다 훨씬 작다** — 앵커를 섞어도 창 내용이 거의 안 바뀐다.")
    run.log("       그러면 G5 셔플 팔은 「정렬이 필요한가」라는 **좁은 주장**만 검정한다")

T2_ = time.time()
for arm in ARMS:
    sh = []
    for rep in range(N_SHUF_G5):
        for rec in COH:
            sr = np.random.RandomState(SEED0 + 7000 + 97 * rep + rec)
            a, _ = run_patient(rec, arm, shuf_anchor=sr, seed=SEED0 + 8000 + 97 * rep + rec)
            sh.append(a)
    m_, lo_, hi_, n_ = boot_mean(sh, SEED0 + 61, NB_BOOT)
    G5[arm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_))
    over = np.isfinite(G4[arm]["hi"]) and lo_ > G4[arm]["hi"]
    run.log(f"  G5 {arm:<14} 셔플 **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · "
            f"영점 {G4[arm]['mean']:.4f} → " + ("⚠️ **영점 초과**" if over else "✅ 영점 안"))
run.log(f"  ({time.time()-T2_:.0f}초 · reps={N_SHUF_G5})")
bad5 = [a for a in ARMS if np.isfinite(G4[a]["hi"]) and G5[a]["lo"] > G4[a]["hi"]]
# ★★ 판정은 **크기**로 한다 — 「CI 가 0.5 를 배제하는가」만 보면 아주 작은 효과가
#    아주 큰 효과의 원인으로 지목된다(1판이 그렇게 틀렸다: G5a 초과 +0.012 로
#    셔플 초과 +0.148 을 설명한다고 찍었다). 검출기 버릇이 **원인이려면** 그 크기가
#    셔플 팔 초과의 상당 부분을 설명해야 한다.
EXC_G5A = float(am - 0.5)
EXC_SHUF = float(G5["raw"]["mean"] - G4["raw"]["mean"])
IDX_SHARE = EXC_G5A / EXC_SHUF if abs(EXC_SHUF) > 1e-9 else float("nan")
IDX_GUILTY = bool(np.isfinite(alo) and alo > 0.5 and np.isfinite(IDX_SHARE)
                  and IDX_SHARE >= 0.5)
run.log("\n  ★ G5 판정은 **G5a 와 함께 · 크기로** 읽는다 — 셔플 팔이 안 떨어지는 이유가 둘이다:")
run.log(f"     (ㄱ) 검출기 버릇을 학습했다 → G5a 초과 {EXC_G5A:+.4f} 가 셔플 초과 "
        f"{EXC_SHUF:+.4f} 의 **{IDX_SHARE:.0%}** 를 설명한다 "
        f"(기준 50% → {'**유죄**' if IDX_GUILTY else '설명 못 한다'})")
run.log(f"     (ㄴ) 앵커 산포({DISP:.1f})가 창({HW_P})보다 작아 셔플이 창 내용을 안 바꾼다")
g_("G5", "✅ 지지" if not bad5 else ("❌ 기각" if IDX_GUILTY else "⚠️ 미결"),
   "앵커를 섞으면 **영점으로 떨어진다** — 모델이 검출기 버릇이 아니라 창 내용을 본다"
   if not bad5 else
   (f"★★ **{', '.join(bad5)} 에서 셔플이 영점을 넘고 G5a 도 영점을 넘는다** — 모델이 "
    "**P 가 아니라 검출기 버릇**을 학습했다. 그 팔의 G1 을 P 형태의 증거로 읽지 않는다"
    if IDX_GUILTY else
    f"⚠️ **{', '.join(bad5)} 에서 셔플이 영점 위에 남지만 검출기 버릇으로는 설명이 안 된다** "
    f"(G5a 초과 {EXC_G5A:+.4f} 가 셔플 초과 {EXC_SHUF:+.4f} 의 {IDX_SHARE:.0%} 뿐). "
    f"앵커 산포({DISP:.1f}샘플)가 창 반폭({HW_P})의 {DISP/HW_P:.0%} 라 셔플이 창 내용을 "
    "거의 안 바꾼 것으로 읽힌다 — 즉 **이 대조는 검정력이 없다**. "
    "**정렬 의존성은 미검정**으로 남기고, 다음 런에서 창을 좁히거나 앵커를 넓게 재배치한다"))

# ── 관측 vs 영점 (팔별)
run.log("\n  관측 대 영점 (팔별 · 초과 = 관측 − 영점)")
EXC = {}
for arm in ARMS:
    exc = G1[arm]["mean"] - G4[arm]["mean"]
    over = np.isfinite(G4[arm]["hi"]) and G1[arm]["lo"] > G4[arm]["hi"]
    EXC[arm] = dict(excess=float(exc), over=bool(over))
    run.log(f"    {arm:<14} 관측 {G1[arm]['mean']:.4f} · 영점 {G4[arm]['mean']:.4f} · "
            f"초과 {exc:+.4f}  " + ("✅ 영점 초과" if over else "⚠️ 영점 안"))
CONFIG["G4"] = G4; CONFIG["G5"] = G5; CONFIG["excess"] = EXC
CONFIG["G5a"] = dict(mean=am, lo=alo, hi=ahi, n=int(an), disp=DISP,
                     disp_frac=float(DISP / HW_P), excess=EXC_G5A,
                     shuf_excess=EXC_SHUF, share=float(IDX_SHARE),
                     idx_guilty=bool(IDX_GUILTY))
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【G-D】 필요표본 · ★ G7 결론 검산표 · 그림 · 요약
run.log("\n" + "=" * 100)
run.log("【G-D】 필요표본 · G7 결론 검산표")
run.log("=" * 100)
run.log(f"  필요표본 (**우월 프레임** · 단위 = 환자 · 현재 {len(COH)}명 · 앵커 {CEIL} 기준)")
NEED = {}
for arm in ARMS:
    d = G1[arm]; eff = d["mean"] - CEIL; half = d["mde"]
    n5 = need_super(len(COH), half, eff, False); n8 = need_super(len(COH), half, eff, True)
    zero = abs(eff) < half
    NEED[arm] = dict(effect=float(eff), half=float(half), sup50=float(n5), sup80=float(n8),
                     uninterpretable=bool(zero))
    run.log(f"  {arm:<14}{eff:>+9.4f}{half:>9.4f}{n5:>9.0f}{n8:>9.0f}  "
            + ("★ **효과 ≈ 0 이라 해석 불가**(R41 ②)" if zero else "읽을 수 있다"))
run.log("    ▸ 판정은 필요표본이 아니라 **MDE 로** 한다")

run.log("\n  ★ G7 — **결론 검산표**")
CHECK = [
    dict(claim=f"천장과 **같은 통계량**으로 쟀다 (잔차화 + f1 층 매칭)",
         num=f"앵커 {CEIL} = Q7-AA AA1 의 CI 상단 · 런 내 `p_score` 기준선 "
             f"{CONFIG['baseline_inrun']['mean']:.4f}",
         assume="평가 구간의 매칭 분포가 Q7-AA 의 전 기록 분포와 **비교 가능**하다는 것",
         iffalse="모집단이 달라 앵커가 헐거우면 **런 내 기준선**이 실질 비교 대상이다 — 둘을 병기했다"),
    dict(claim=f"G1 최량 팔 {BEST} {G1[BEST]['mean']:.4f} [{G1[BEST]['lo']:.4f}, {G1[BEST]['hi']:.4f}]",
         num=f"팔별 판정을 전부 찍었다 · MDE {G1[BEST]['mde']:.4f} · 환자 {len(COH)}명",
         assume="**최량 팔은 사후 선택**이다 — 그래서 종결 판단은 **어느 팔도 못 넘을 때**만 한다",
         iffalse="한 팔만 보고 통과를 선언하면 3팔 중 최대만큼 낙관 편향된다(R36 ②)"),
    dict(claim=f"영점을 **측정**했다 (파이프라인 영점 · reps={N_PERM_G4})",
         num=" · ".join(f"{a} {G4[a]['mean']:.4f}" for a in ARMS),
         assume="**없음** — 학습 라벨만 치환하고 같은 절차를 재학습한다",
         iffalse="—"),
    # ★ claim 이 **판정과 같은 세 갈래**여야 한다 — 요약이 셀과 어긋나면 안 된다(R38 ⑦)
    dict(claim=("G5 앵커 셔플이 영점으로 떨어진다" if not bad5 else
                (f"★★ G5 위반(검출기 버릇) — {', '.join(bad5)}" if IDX_GUILTY else
                 f"⚠️ G5 미결(대조 검정력 없음) — {', '.join(bad5)}")),
         num=" · ".join(f"{a} {G5[a]['mean']:.4f}(영점 {G4[a]['mean']:.4f})" for a in ARMS),
         assume=f"셔플이 창 내용을 **실제로 바꾼다**는 것 — 앵커 산포 {DISP:.1f}샘플 vs "
                f"창 반폭 {HW_P}샘플({DISP/HW_P:.0%})",
         iffalse="산포가 창보다 작으면 셔플은 창 내용을 거의 안 바꾸므로 **좁은 주장만** "
                 f"검정한다. 그래서 **G5a**(학습 없이 `p_idx` 단독) {am:.4f} "
                 f"[{alo:.4f}, {ahi:.4f}] 를 함께 뒀다 — 검출기 버릇의 직접 검정이다"),
    dict(claim=f"G5a — `p_idx` 단독 매칭 AUROC {am:.4f} [{alo:.4f}, {ahi:.4f}]",
         num=f"학습 없음 · 앵커 산포 중앙 {DISP:.1f}샘플 · 환자 {an}명",
         assume="**없음** — 앵커 값만으로 잰다",
         iffalse=f"이게 영점을 넘으면 **검출기가 라벨과 상관**된다는 뜻이다. 다만 판정은 "
                 f"**크기로** 한다 — 초과 {EXC_G5A:+.4f} 가 셔플 초과 {EXC_SHUF:+.4f} 의 "
                 f"{IDX_SHARE:.0%} 다. 「CI 가 0.5 를 배제」만 보면 아주 작은 효과가 아주 큰 "
                 "효과의 원인으로 지목된다"),
    dict(claim="소거 팔의 교란을 음성 대조로 막았다",
         num=f"`cancel − cancel_pmask` {PAIRS['cancel−cancel_pmask']['mean']:+.4f} "
             f"[{PAIRS['cancel−cancel_pmask']['lo']:+.4f}, "
             f"{PAIRS['cancel−cancel_pmask']['hi']:+.4f}]",
         assume="P 봉우리 ±{}ms 마스킹이 P 정보를 **충분히** 제거한다는 것".format(int(PMASK_MS)),
         iffalse="마스크가 좁으면 잔여 P 가 남아 대조가 헐거워진다 — 그 경우 이 차는 **하한**이다"),
    dict(claim="누출을 막았다",
         num=f"시간 분할 앞{FRAC_TRAIN:.0%}/뒤 · 가드밴드 {GUARD_S:.0f}초 · "
             "소거 템플릿도 **학습 구간에서만** · 표준화는 학습 통계로만",
         assume="**없음** — 구성으로 보장된다",
         iffalse="—"),
    dict(claim="용량 선택 편의가 없다",
         num=f"고정 CNN 하나(약 740 파라미터) · EPOCHS={EPOCHS} 고정 · 격자 없음",
         assume="**없음** — 선택이 없으면 선택 편의도 없다(R22 · R36 ②)",
         iffalse="—  ★ 용량 격자는 G1 을 넘은 뒤 **G2 에서 LORO/DEV 반쪽**으로"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["need"] = NEED; CONFIG["G7"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【G-E】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례·제목은 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
EN = {"raw": "raw", "cancel": "cancel", "cancel_pmask": "cancel+Pmask\n(neg. control)"}

# ① G1 — 팔별 관측 vs 영점 vs 앵커
xs = np.arange(len(ARMS))
obs = [G1[a]["mean"] for a in ARMS]
oe = [[obs[i] - G1[a]["lo"] for i, a in enumerate(ARMS)],
      [G1[a]["hi"] - obs[i] for i, a in enumerate(ARMS)]]
nul = [G4[a]["mean"] for a in ARMS]
ne = [[nul[i] - G4[a]["lo"] for i, a in enumerate(ARMS)],
      [G4[a]["hi"] - nul[i] for i, a in enumerate(ARMS)]]
ax[0].errorbar(xs - 0.12, obs, yerr=oe, fmt="o", capsize=5, color="tab:red", label="observed")
ax[0].errorbar(xs + 0.12, nul, yerr=ne, fmt="x", capsize=4, color="tab:gray",
               label="pipeline null")
ax[0].axhline(CEIL, ls="--", color="tab:blue", lw=1.2, label=f"entry anchor {CEIL}")
ax[0].axhline(CONFIG["baseline_inrun"]["mean"], ls=":", color="tab:green", lw=1.2,
              label=f"in-run p_score {CONFIG['baseline_inrun']['mean']:.3f}")
ax[0].axhline(0.5, color="k", lw=.8)
ax[0].set_xticks(xs); ax[0].set_xticklabels([EN[a] for a in ARMS], fontsize=8)
ax[0].set_ylabel("matched AUROC (rhythm-residualised)")
ax[0].set_title("G1 : entry gate", fontsize=9)
ax[0].legend(fontsize=6); ax[0].grid(alpha=.3, axis="y")

# ② G5 — 앵커 셔플이 영점으로 떨어지나
sh = [G5[a]["mean"] for a in ARMS]
se = [[sh[i] - G5[a]["lo"] for i, a in enumerate(ARMS)],
      [G5[a]["hi"] - sh[i] for i, a in enumerate(ARMS)]]
ax[1].errorbar(xs - 0.12, sh, yerr=se, fmt="s", capsize=5, color="tab:purple",
               label="p_idx shuffled (G5)")
ax[1].errorbar(xs + 0.12, nul, yerr=ne, fmt="x", capsize=4, color="tab:gray",
               label="pipeline null (G4)")
ax[1].axhline(0.5, color="k", lw=.8)
ax[1].set_xticks(xs); ax[1].set_xticklabels([EN[a] for a in ARMS], fontsize=8)
ax[1].set_ylabel("matched AUROC")
ax[1].set_title("G5 : ruler artefact control", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="y")

# ③ 환자별 산점 — raw vs cancel
pr_ = [G1["raw"]["per"][r] for r in COH]
pc_ = [G1["cancel"]["per"][r] for r in COH]
ax[2].scatter(pr_, pc_, s=30, color="tab:orange")
lim = [min(pr_ + pc_ + [0.4]) - 0.02, max(pr_ + pc_ + [0.7]) + 0.02]
ax[2].plot(lim, lim, "k--", lw=.9)
ax[2].axhline(CEIL, ls=":", color="tab:blue", lw=1.0)
ax[2].axvline(CEIL, ls=":", color="tab:blue", lw=1.0)
ax[2].set_xlabel("raw arm (per patient)"); ax[2].set_ylabel("cancel arm (per patient)")
ax[2].set_title(f"per-patient, n={len(COH)}", fontsize=9)
ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q8_g1_personal_pmorph", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:4]:
    run.log(f"  {g:<5}{VERD.get(g, '(미실행)')}")
run.log("")
run.log(f"  환자 {len(COH)}명 · 앵커 {CEIL} · 런 내 `p_score` 기준선 "
        f"{CONFIG['baseline_inrun']['mean']:.4f}")
for a in ARMS:
    run.log(f"    {a:<14}{G1[a]['mean']:.4f} [{G1[a]['lo']:.4f}, {G1[a]['hi']:.4f}] · "
            f"영점 {G4[a]['mean']:.4f} · 셔플 {G5[a]['mean']:.4f}")
run.log("")
# ★ 요약 분기가 **판정과 같은 세 갈래**여야 한다 — 검산표만 고치고 요약을 안 고쳐서
#   1판 공식 실행 로그에 「G5 ⚠️ 미결」과 「⛔ G5 위반」이 **함께 찍혔다**(R38 ⑦).
if bad5 and IDX_GUILTY:
    run.log(f"  ⛔ **G5 위반(검출기 버릇) — {', '.join(bad5)}**: 앵커를 섞어도 영점을 넘고")
    run.log(f"     G5a 초과 {EXC_G5A:+.4f} 가 셔플 초과 {EXC_SHUF:+.4f} 의 {IDX_SHARE:.0%} 를 설명한다.")
    run.log("     그 팔은 **P 형태가 아니라 검출기 버릇**을 학습했다 — G1 수치를 인용하지 않는다")
elif bad5:
    run.log(f"  ⚠️ **G5 미결 — {', '.join(bad5)} 에서 셔플이 영점 위에 남는다.**")
    run.log(f"     단 검출기 버릇으로는 설명이 안 된다(G5a 초과 {EXC_G5A:+.4f} = 셔플 초과의 "
            f"{IDX_SHARE:.0%}). 앵커 산포 {DISP:.1f}샘플이 창 반폭 {HW_P}의 {DISP/HW_P:.0%} 라")
    run.log("     **이 대조에 검정력이 없다** — 정렬 의존성은 미검정으로 남는다.")
    run.log("     ★ 그런데 셔플해도 성능이 안 떨어진다는 건 **신호가 정렬에 의존하지 않는다**는 뜻이고,")
    run.log("       P 형태라면 그럴 수 없다. 창 전체에 퍼진 무언가일 가능성을 다음 런에서 가른다")
elif ok_("G1"):
    run.log(f"  ★★★ **G1 통과 — 학습 표현이 천장을 넘는다** (최량 {BEST} "
            f"{G1[BEST]['mean']:.4f} > {CEIL})")
    run.log("     → Q8 진입. 다음은 **G2 라벨-이득 곡선**(임상 결정 숫자는 AUROC 가 아니라 **N**)")
    run.log("       그리고 **G3 개인화 리듬-only 기저** — 이득이 P 형태의 것인지 개인화 자체의")
    run.log("       것인지는 G3 없이 못 가른다(R24 · R27)")
    if PAIRS["cancel−raw"]["lo"] > 0:
        run.log(f"     ★ 소거가 이득을 준다 — `cancel − raw` "
                f"{PAIRS['cancel−raw']['mean']:+.4f} "
                f"[{PAIRS['cancel−raw']['lo']:+.4f}, {PAIRS['cancel−raw']['hi']:+.4f}]. "
                "**판별력에서 소거가 처음 살아난 것이다**")
else:
    run.log(f"  ⛔⛔ **G1 미달 — 어느 팔도 앵커 {CEIL} 을 못 넘는다. Q8 을 종결한다.**")
    run.log("     최종 문장:")
    run.log("       「2리드 홀터 SVEB 검출에서, 환자 안 시간 분할로 **그 환자의 라벨로 학습한**")
    run.log(f"        P 창 표현의 리듬 통제 후 매칭 AUROC 는 {G1[BEST]['mean']:.4f} ")
    run.log(f"        [{G1[BEST]['lo']:.4f}, {G1[BEST]['hi']:.4f}] 로, 1차원 손수 특징 `p_score`")
    run.log(f"        의 천장 CI 상단({CEIL})을 넘지 못한다. **개인화라는 이름으로 되살리지 않는다.**」")
    run.log("     ★ 그리고 **QRST 소거도 판별력에서 처음이자 마지막으로 측정됐다** — "
            f"`cancel − raw` {PAIRS['cancel−raw']['mean']:+.4f} "
            f"[{PAIRS['cancel−raw']['lo']:+.4f}, {PAIRS['cancel−raw']['hi']:+.4f}]")
run.log("")
run.log("  ▸ ★ 최량 팔은 **사후 선택**이다 — 팔별 판정을 전부 찍었다(R36 ②)")
run.log("  ▸ ★ T2 의 가시성(+0.2825)은 **판별력의 근거가 아니다**(R40 ①)")
run.log("  ▸ ★ G2·G3·G6 은 **G1 을 넘은 뒤에** 짓는다")

run.finish({
    "exp_id": "quest46_q8_g1_personal_pmorph",
    "metric": "matched_auroc_best_arm",
    "value": float(G1[BEST]["mean"]),
    "passed": bool(ok_("G0") and ok_("G1") and ok_("G5")),
    "summary": ("Q8 진입 관문 G1 — 환자 안 시간 분할에서 학습 표현(고정 소형 1D CNN)의 "
                "리듬 잔차화·매칭 AUROC 가 p_score 천장 0.6097 을 넘는가. 3팔(raw · QRST 소거 · "
                "소거+P마스크 음성 대조). 소거는 판별력에서 처음 측정된다. "
                "파이프라인 영점(학습 라벨 치환 후 재학습)과 p_idx 셔플 대조를 함께 잰다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "G0": CONFIG.get("G0", {}), "G1": CONFIG.get("G1", {}), "G1_per": CONFIG.get("G1_per", {}),
    "G1_pairs": CONFIG.get("G1_pairs", {}), "G4": CONFIG.get("G4", {}),
    "G5": CONFIG.get("G5", {}), "G5a": CONFIG.get("G5a", {}),
    "excess": CONFIG.get("excess", {}),
    "baseline_inrun": CONFIG.get("baseline_inrun", {}), "need": CONFIG.get("need", {}),
    "G7": CONFIG.get("G7", []), "cohort": COH, "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q8_g1_personal_pmorph.ipynb`")
